# Витрина транзакций ТЭ — 5. Профиль клиента

AI-14135. Одна строка = один ИНН. Таблица `sandbox_ai.tmp_shestopalov_te_profile`.

Сначала должны быть собраны факт `01_te_trx.ipynb` и клиент `02_te_client.ipynb`. Терминалы не нужны.

Профиль читает уже собранный факт, не `scd1_trx`. У ИНН сохраняются все коды чека (`mcc_trx_codes`) и все коды точки (`mcc_terminal_codes`). Для разреза по-прежнему берётся самый частый код точки по успешным покупкам (статус не `R`). Профиль считается за всё окно факта, август–октябрь 2026. Регион в разрезе — регион регистрации: `04` Красноярский край, `45` Москва, остальные — «прочие». Размер бизнеса — зона ответственности. Топ-10 MCC точки по обороту остаются отдельными кодами, остальные склеиваются по профилю точки.

Пары «код точки / код чека» в профиль не входят. Они выводятся отдельным запросом в последней ячейке.

`p10` и `p90` Impala здесь не считает. Их добавляет `te_trx_analysis.py`, если на вход дан факт, а не этот профиль.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
with imp:
    keys = imp.fetch("""
select f.fact_rows, f.fact_inn, c.client_rows
from (
  select count(*) as fact_rows, count(distinct inn) as fact_inn
  from sandbox_ai.tmp_shestopalov_acq_trx
) f
cross join (
  select count(*) as client_rows
  from sandbox_ai.tmp_shestopalov_acq_clients
) c
""")
display(keys)
assert keys.loc[0, "fact_rows"] > 0, "Факт пуст: сначала выполните 01_te_trx.ipynb"
assert keys.loc[0, "client_rows"] > 0, "Клиент пуст: сначала выполните 02_te_client.ipynb"


In [ ]:
with imp:
    imp.execute("set MEM_LIMIT=12g")
    imp.execute("drop table if exists sandbox_ai.tmp_shestopalov_te_profile")
    imp.execute("""
create table sandbox_ai.tmp_shestopalov_te_profile
as
with trx as (
  select
    inn,
    transaction_id,
    sum_t,
    date_t,
    card_bin,
    payment_system,
    mcc_code,
    mcc_terminal,
    nullif(trim(mcc_profile), '') as mcc_profile,
    mcc_mismatch,
    case when coalesce(status, '') <> 'R' then 1 else 0 end as is_success,
    unix_timestamp(date_t) - unix_timestamp(
      lag(date_t) over (partition by inn order by date_t, transaction_id)
    ) as time_diff_sec
  from sandbox_ai.tmp_shestopalov_acq_trx
),
agg as (
  select
    inn,
    count(*) as tx_count,
    sum(sum_t) as turnover,
    avg(sum_t) as avg_amount,
    appx_median(sum_t) as median_amount,
    stddev_samp(sum_t) as std_amount,
    avg(time_diff_sec) as avg_interval,
    appx_median(time_diff_sec) as median_interval,
    avg(cast(is_success as double)) as success_rate,
    count(distinct to_date(date_t)) as active_days,
    count(distinct card_bin) as unique_bins,
    count(distinct payment_system) as unique_card_types,
    avg(cast(mcc_mismatch as double)) as mcc_mismatch_share
  from trx
  group by inn
),
mcc_freq as (
  select
    inn,
    mcc_terminal,
    count(*) as cnt
  from trx
  where is_success = 1
    and mcc_terminal is not null
  group by inn, mcc_terminal
),
mcc_mode as (
  select inn, mcc_terminal as mcc_code
  from (
    select
      inn,
      mcc_terminal,
      row_number() over (
        partition by inn
        order by cnt desc, mcc_terminal
      ) as rn
    from mcc_freq
  ) z
  where rn = 1
),
profile_turn as (
  select
    mcc_terminal,
    coalesce(mcc_profile, '') as mcc_profile,
    sum(sum_t) as turnover
  from trx
  where is_success = 1
    and mcc_terminal is not null
  group by mcc_terminal, coalesce(mcc_profile, '')
),
profile_mode as (
  select mcc_terminal, mcc_profile
  from (
    select
      mcc_terminal,
      mcc_profile,
      row_number() over (
        partition by mcc_terminal
        order by turnover desc, mcc_profile
      ) as rn
    from profile_turn
  ) z
  where rn = 1
),
mcc_turn as (
  select mcc_terminal, sum(turnover) as turnover
  from profile_turn
  group by mcc_terminal
),
top10 as (
  select mcc_terminal
  from (
    select
      mcc_terminal,
      row_number() over (order by turnover desc, mcc_terminal) as rn
    from mcc_turn
  ) z
  where rn <= 10
),
mcc_grp as (
  select
    t.mcc_terminal,
    case
      when top.mcc_terminal is not null then t.mcc_terminal
      when p.mcc_profile is null or p.mcc_profile = '' then t.mcc_terminal
      else p.mcc_profile
    end as mcc_group
  from mcc_turn t
  left join top10 top
    on top.mcc_terminal = t.mcc_terminal
  left join profile_mode p
    on p.mcc_terminal = t.mcc_terminal
),
mcc_lists as (
  select
    inn,
    group_concat(distinct mcc_code, ',') as mcc_trx_codes,
    group_concat(distinct mcc_terminal, ',') as mcc_terminal_codes
  from trx
  group by inn
)
select
  a.inn,
  a.tx_count,
  a.turnover,
  a.avg_amount,
  a.median_amount,
  a.std_amount,
  a.avg_interval,
  a.median_interval,
  a.success_rate,
  a.active_days,
  a.unique_bins,
  a.unique_card_types,
  a.mcc_mismatch_share,
  coalesce(lst.mcc_trx_codes, 'не заполнен') as mcc_trx_codes,
  coalesce(lst.mcc_terminal_codes, 'не заполнен') as mcc_terminal_codes,
  coalesce(m.mcc_code, 'не заполнен') as mcc_code,
  coalesce(g.mcc_group, coalesce(m.mcc_code, 'не заполнен')) as mcc_group,
  coalesce(c.region, 'не заполнен') as region,
  case when c.region in ('04', '45') then c.region else 'прочие' end as region_group,
  coalesce(c.business_size, 'нет ЗО') as business_size,
  coalesce(c.business_size_is_proxy, cast(0 as tinyint)) as business_size_is_proxy,
  coalesce(c.okved, 'не заполнен') as okved,
  case
    when c.okved is null or trim(c.okved) = '' then 'не заполнен'
    else substring(trim(c.okved), 1, 2)
  end as okved_class,
  coalesce(c.okopf, 'НЕИЗВЕСТНО') as okopf,
  c.date_acquiring,
  concat(
    coalesce(m.mcc_code, 'не заполнен'),
    ' | ',
    case when c.region in ('04', '45') then c.region else 'прочие' end
  ) as slice_mcc_region,
  concat(
    coalesce(m.mcc_code, 'не заполнен'),
    ' | ',
    coalesce(c.business_size, 'нет ЗО')
  ) as slice_mcc_size
from agg a
left join mcc_lists lst
  on lst.inn = a.inn
left join mcc_mode m
  on m.inn = a.inn
left join mcc_grp g
  on g.mcc_terminal = m.mcc_code
left join sandbox_ai.tmp_shestopalov_acq_clients c
  on c.inn = a.inn
""")
print("собрано sandbox_ai.tmp_shestopalov_te_profile")


In [ ]:
with imp:
    rows = imp.fetch("""
select count(*) as rows_cnt, count(distinct inn) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_profile
""")
    display(rows)
    assert rows.loc[0, "rows_cnt"] == rows.loc[0, "inn_cnt"], "В профиле дубли inn"
    assert rows.loc[0, "inn_cnt"] == keys.loc[0, "fact_inn"], "В профиле не столько ИНН, сколько в факте"

    display(imp.fetch("""
select region_group, count(*) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_profile
group by region_group
order by inn_cnt desc
"""))
    display(imp.fetch("""
select business_size, count(*) as inn_cnt
from sandbox_ai.tmp_shestopalov_te_profile
group by business_size
order by inn_cnt desc
"""))
    display(imp.fetch("""
select inn, mcc_trx_codes, mcc_terminal_codes, mcc_code, mcc_group
from sandbox_ai.tmp_shestopalov_te_profile
order by tx_count desc
limit 20
"""))
    display(imp.fetch("""
select *
from sandbox_ai.tmp_shestopalov_te_profile
limit 20
"""))


In [ ]:
with imp:
    display(imp.fetch("""
select
  mcc_terminal,
  mcc_code,
  count(*) as tx_cnt,
  sum(sum_t) as turnover
from sandbox_ai.tmp_shestopalov_acq_trx
where mcc_mismatch = 1
group by mcc_terminal, mcc_code
order by tx_cnt desc
"""))
